# B1.3 · Patch generation and local validation

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

---

**Risk.** Unverified agent patches destroy trust faster than missed findings.

**Control.** Build and test the fix locally *before* raising the PR.

**This lab.** Never raise a PR for a patch you did not build and test.

| | |
|---|---|
| Open-source tooling | Semgrep Autofix, pytest |
| Open-weight models | Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B1.3"))

A patch is only a patch if the test that proves it exists. Otherwise an agent can 'fix' a bug by deleting the code path and the harness will applaud.

In [ ]:
from cybercommons import appsec

before = appsec.scan("sql_injection", appsec.SNIPPETS["sql_injection"])
f = before[0]
print("finding:", f.key(), "—", f.evidence, "\n")

patched = appsec.SNIPPETS["safe_parameterised"]
after = appsec.scan("sql_injection", patched)
print("after the patch, findings:", after or "none")

for label, p in (("no regression test", appsec.Patch(f.key(), "diff", test_added=False)),
                 ("with test",         appsec.Patch(f.key(), "diff", test_added=True))):
    ok, why = p.validate(after)
    print(f"  {label:20s} accepted={str(ok):5s} — {why}")

Now the failure mode worth naming: the finding disappears but the code is not fixed.

In [ ]:
deleted = appsec.Patch(f.key(), "removed the function entirely", test_added=False)
print(deleted.validate([]))
print("\nThe finding is gone. Nothing proves the behaviour is preserved.")
print("Local validation has to assert what still WORKS, not only what stopped firing.")

### Expect

The rescan of the parameterised version returns no findings. The untested patch is rejected for having no regression test; the tested one is accepted. Deleting the function is also rejected.

### Your turn

Add a third clause to `validate`: the original functional tests must still pass. Which of the three clauses is hardest to get in a real repository, and why is that the interesting one?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B1.3.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*